In [ ]:
from scipy.optimize import minimize
from scipy.integrate import odeint
import numpy as np
import matplotlib.pyplot as plt

# ── MODELO ODE ───────────────────────────────────────────────────────────────
def modelo(y, t, k_Au, k_As, Rmax_Au, Rmax_As):
    R_Au, R_As = y
    dR_Audt = k_Au * (Rmax_Au - R_Au)
    dR_Asdt = k_As * (Rmax_As - R_As)
    return [dR_Audt, dR_Asdt]

# ── PARÁMETROS CINÉTICOS ─────────────────────────────────────────────────────
y0      = [0, 0]
Rmax_Au = 85
Rmax_As = 70
k_Au    = 0.08   # min⁻¹ — literatura sulfuros (Xu, 1998)
k_As    = 0.03   # min⁻¹ — literatura sulfuros (Xu, 1998)

# ── FUNCIÓN AUXILIAR ─────────────────────────────────────────────────────────
def calcular_R_Au(vars):
    t1, t2, t3 = vars
    s1 = odeint(modelo, y0,      [0, t1], args=(k_Au, k_As, Rmax_Au, Rmax_As))
    s2 = odeint(modelo, s1[-1],  [0, t2], args=(k_Au, k_As, Rmax_Au, Rmax_As))
    s3 = odeint(modelo, s2[-1],  [0, t3], args=(k_Au, k_As, Rmax_Au, Rmax_As))
    return s3[-1][0]

# ── FUNCIÓN OBJETIVO ─────────────────────────────────────────────────────────
def objetivo(vars):
    t1, t2, t3 = vars
    s1 = odeint(modelo, y0,      [0, t1], args=(k_Au, k_As, Rmax_Au, Rmax_As))
    s2 = odeint(modelo, s1[-1],  [0, t2], args=(k_Au, k_As, Rmax_Au, Rmax_As))
    s3 = odeint(modelo, s2[-1],  [0, t3], args=(k_Au, k_As, Rmax_Au, Rmax_As))
    return -(s3[-1][0] - 1.5 * s3[-1][1])

# ── RESTRICCIONES ────────────────────────────────────────────────────────────
constraints = [
    {'type': 'ineq', 'fun': lambda v: v[0] - 5},   # t1 >= 5 min
    {'type': 'ineq', 'fun': lambda v: 15 - v[0]},  # t1 <= 15 min
    {'type': 'ineq', 'fun': lambda v: v[1] - 5},   # t2 >= 5 min
    {'type': 'ineq', 'fun': lambda v: 15 - v[1]},  # t2 <= 15 min
    {'type': 'ineq', 'fun': lambda v: v[2] - 5},   # t3 >= 5 min
    {'type': 'ineq', 'fun': lambda v: 15 - v[2]},  # t3 <= 15 min
    {'type': 'ineq', 'fun': lambda v: calcular_R_Au(v) - 70},  # R_Au >= 70%
]

# ── OPTIMIZACIÓN ─────────────────────────────────────────────────────────────
resultado   = minimize(objetivo, x0=[8, 4, 2], constraints=constraints)
t1_opt, t2_opt, t3_opt = resultado.x

# Extraer R_Au y R_As en el óptimo
s1_opt = odeint(modelo, y0,          [0, t1_opt], args=(k_Au, k_As, Rmax_Au, Rmax_As))
s2_opt = odeint(modelo, s1_opt[-1],  [0, t2_opt], args=(k_Au, k_As, Rmax_Au, Rmax_As))
s3_opt = odeint(modelo, s2_opt[-1],  [0, t3_opt], args=(k_Au, k_As, Rmax_Au, Rmax_As))

R_Au_pct = s3_opt[-1][0]   # % — recuperación Au final
R_As_pct = s3_opt[-1][1]   # % — recuperación As final

print("=" * 52)
print("    OPTIMIZACIÓN — CIRCUITO ROUGHER FLOATSIM")
print("=" * 52)
print(f"  k_Au:           {k_Au} min⁻¹")
print(f"  k_As:           {k_As} min⁻¹")
print(f"  Rmax_Au:        {Rmax_Au}%")
print(f"  Rmax_As:        {Rmax_As}%")
print(f"  R_Au mínimo:    70%  (restricción)")
print("-" * 52)
print(f"  t1 óptimo:      {t1_opt:.2f} min")
print(f"  t2 óptimo:      {t2_opt:.2f} min")
print(f"  t3 óptimo:      {t3_opt:.2f} min")
print(f"  R_Au final:     {R_Au_pct:.2f}%")
print(f"  R_As final:     {R_As_pct:.2f}%")
print(f"  F (selectividad): {-resultado.fun:.2f}")
print("=" * 52)

# ── GRÁFICA ───────────────────────────────────────────────────────────────────
t1_c = np.linspace(0, t1_opt, 100)
t2_c = np.linspace(0, t2_opt, 100)
t3_c = np.linspace(0, t3_opt, 100)

sol1_g = odeint(modelo, [0, 0],        t1_c, args=(k_Au, k_As, Rmax_Au, Rmax_As))
sol2_g = odeint(modelo, sol1_g[-1],    t2_c, args=(k_Au, k_As, Rmax_Au, Rmax_As))
sol3_g = odeint(modelo, sol2_g[-1],    t3_c, args=(k_Au, k_As, Rmax_Au, Rmax_As))

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
titulos   = [f'Celda 1 — t={t1_opt:.1f} min',
             f'Celda 2 — t={t2_opt:.1f} min',
             f'Celda 3 — t={t3_opt:.1f} min']
tiempos   = [t1_c,   t2_c,   t3_c]
soluciones= [sol1_g, sol2_g, sol3_g]

for i, ax in enumerate(axes):
    ax.plot(tiempos[i], soluciones[i][:, 0], color='gold', lw=2, label='Au')
    ax.plot(tiempos[i], soluciones[i][:, 1], color='gray', lw=2, label='As')
    ax.set_title(titulos[i], fontsize=11)
    ax.set_xlabel('Tiempo (min)')
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[0].set_ylabel('Recuperación (%)')
plt.suptitle('Cinética de flotación — Circuito Rougher Optimizado\nArsenopirita Aurífera',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── BALANCE DE MASA ──────────────────────────────────────────────────────────
feed        = 100               # t/h
ley_Au_feed = 10                # g/t
pct_As_feed = 4                 # %
R_Au        = R_Au_pct / 100    # fracción — viene del optimizador
R_As        = R_As_pct / 100    # fracción — viene del optimizador
pct_conc    = 0.05              # fracción másica al concentrado

# Flujos másicos
concentrado = feed * pct_conc
relave      = feed - concentrado

# Balance de Au
Au_feed        = feed * ley_Au_feed
Au_concentrado = Au_feed * R_Au
Au_relave      = Au_feed - Au_concentrado
ley_Au_conc    = Au_concentrado / concentrado
ley_Au_relave  = Au_relave / relave

# Balance de As
As_feed        = feed * (pct_As_feed / 100)
As_concentrado = As_feed * R_As
As_relave      = As_feed - As_concentrado
pct_As_conc    = (As_concentrado / concentrado) * 100
pct_As_relave  = (As_relave / relave) * 100

print("\n" + "=" * 52)
print("    BALANCE DE MASA — CIRCUITO ROUGHER FLOATSIM")
print("=" * 52)
print(f"{'Corriente':<15} {'Flujo (t/h)':>12} {'Au (g/t)':>10} {'As (%)':>8}")
print("-" * 52)
print(f"{'Feed':<15} {feed:>12.1f} {ley_Au_feed:>10.1f} {pct_As_feed:>8.1f}")
print(f"{'Concentrado':<15} {concentrado:>12.1f} {ley_Au_conc:>10.1f} {pct_As_conc:>8.1f}")
print(f"{'Relave':<15} {relave:>12.1f} {ley_Au_relave:>10.2f} {pct_As_relave:>8.1f}")
print("=" * 52)
print(f"\n  Recuperación Au:    {R_Au*100:.1f}%")
print(f"  Recuperación As:    {R_As*100:.1f}%")
print(f"  Enriquecimiento Au: {ley_Au_conc/ley_Au_feed:.1f}x")